## Etapa 1
Testar diferentes modelos prontos de processamento visual, o intuito é descobrir qual modelo melhor generaliza e se enquadra na solução do problema (identificar características).

- ResNet18
- DenseNet
- VisionTransformer
- ConvNeXT
- EfficientNet-B0
- EfficientNet-B1

In [1]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import f1_score, balanced_accuracy_score

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


In [ ]:
import os
from dotenv import load_dotenv


print("Env carregado com sucesso.") if load_dotenv() else print("Erro ao carregar env")


DATA_DIR = os.getenv("DATA_PATH")
BATCH_SIZE = 32

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

full_ds = ImageFolder(DATA_DIR, transform=train_tf)
num_classes = len(full_ds.classes)

n = len(full_ds)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

# val/test sem augmentation
val_ds.dataset.transform = eval_tf
test_ds.dataset.transform = eval_tf

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print("Classes:", full_ds.classes)
print(f"Split -> train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

Env carregado com sucesso.
Classes: ['Healthy', 'Rust_Blight', 'Rust_Common']
Split -> train=2051 val=439 test=441


### ResNet18

In [ ]:
def build_model(model_name: str, num_classes: int):
    if model_name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_f = m.fc.in_features
        m.fc = nn.Linear(in_f, num_classes)
        head_params = m.fc.parameters()
        last_block_params = m.layer4.parameters()
    elif model_name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        in_f = m.classifier.in_features
        m.classifier = nn.Linear(in_f, num_classes)
        head_params = m.classifier.parameters()
        last_block_params = m.features.denseblock4.parameters()
    elif model_name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_f = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_f, num_classes)
        head_params = m.classifier.parameters()
        last_block_params = m.features[-1].parameters()
    elif model_name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        in_f = m.classifier[2].in_features
        m.classifier[2] = nn.Linear(in_f, num_classes)
        head_params = m.classifier.parameters()
        last_block_params = m.features[-1].parameters()
    elif model_name == "vit_b_16":
        m = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        in_f = m.heads.head.in_features
        m.heads.head = nn.Linear(in_f, num_classes)
        head_params = m.heads.parameters()
        last_block_params = m.encoder.layers[-1].parameters()
    else:
        raise ValueError(f"Modelo não suportado: {model_name}")

    return m, head_params, last_block_params


def set_train_mode(model, mode: str):
    # congela tudo
    for p in model.parameters():
        p.requires_grad = False

    # linear_probe = only head
    # finetune_partial = head + last block
    if mode == "linear_probe":
        keywords = ["fc", "classifier", "heads"]
    elif mode == "finetune_partial":
        keywords = ["fc", "classifier", "heads", "layer4", "denseblock4", "features.7", "features.8", "encoder.layers.11"]
    else:
        raise ValueError("mode deve ser 'linear_probe' ou 'finetune_partial'")

    for n, p in model.named_parameters():
        if any(k in n for k in keywords):
            p.requires_grad = True

In [5]:
# 3) TREINO / VALIDAÇÃO / TESTE
def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()

    total_loss, y_true, y_pred = 0.0, [], []

    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = (np.array(y_true) == np.array(y_pred)).mean()
    f1m = f1_score(y_true, y_pred, average="macro")
    bacc = balanced_accuracy_score(y_true, y_pred)
    return avg_loss, acc, f1m, bacc


def train_model(model, train_loader, val_loader, epochs=12, lr=1e-3, weight_decay=1e-4):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_state, best_val_f1 = None, -1.0

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc, tr_f1, tr_bacc = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc, va_f1, va_bacc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()  # 1x por época

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            best_state = copy.deepcopy(model.state_dict())

        print(f"[{ep:02d}/{epochs}] "
              f"train loss={tr_loss:.4f} acc={tr_acc:.3f} f1={tr_f1:.3f} | "
              f"val loss={va_loss:.4f} acc={va_acc:.3f} f1={va_f1:.3f} bacc={va_bacc:.3f}")

    model.load_state_dict(best_state)
    return model


def evaluate_test(model, test_loader):
    criterion = nn.CrossEntropyLoss()
    te_loss, te_acc, te_f1, te_bacc = run_epoch(model, test_loader, criterion, optimizer=None)
    return {"test_loss": te_loss, "test_acc": te_acc, "test_f1_macro": te_f1, "test_bacc": te_bacc}

In [ ]:
models_to_test = ["resnet18", "densenet121", "efficientnet_b0", "convnext_tiny", "vit_b_16"]
results = []

for model_name in models_to_test:
    for mode in ["linear_probe", "finetune_partial"]:
        print(f"\n=== {model_name} | {mode} ===")
        model, _, _ = build_model(model_name, num_classes)
        set_train_mode(model, mode)

        lr = 1e-3 if mode == "linear_probe" else 1e-4
        epochs = 8 if mode == "linear_probe" else 12

        model = train_model(model, train_loader, val_loader, epochs=epochs, lr=lr)
        metrics = evaluate_test(model, test_loader)

        row = {"model": model_name, "mode": mode, **metrics}
        results.append(row)
        print("TEST:", row)

# ranking simples por F1 macro
results = sorted(results, key=lambda x: x["test_f1_macro"], reverse=True)
print("\nRanking (F1 macro):")
for r in results:
    print(r)


=== resnet18 | linear_probe ===
[01/8] train loss=0.6549 acc=0.750 f1=0.749 | val loss=0.4234 acc=0.872 f1=0.871 bacc=0.882
[02/8] train loss=0.3555 acc=0.887 f1=0.887 | val loss=0.2908 acc=0.913 f1=0.915 bacc=0.918
[03/8] train loss=0.2864 acc=0.911 f1=0.911 | val loss=0.2729 acc=0.918 f1=0.920 bacc=0.922
[04/8] train loss=0.2682 acc=0.910 f1=0.910 | val loss=0.2786 acc=0.902 f1=0.903 bacc=0.909
[05/8] train loss=0.2430 acc=0.919 f1=0.919 | val loss=0.2315 acc=0.925 f1=0.926 bacc=0.929
[06/8] train loss=0.2400 acc=0.919 f1=0.919 | val loss=0.2268 acc=0.925 f1=0.927 bacc=0.928
[07/8] train loss=0.2291 acc=0.929 f1=0.929 | val loss=0.2352 acc=0.913 f1=0.915 bacc=0.919
[08/8] train loss=0.2240 acc=0.922 f1=0.922 | val loss=0.2362 acc=0.911 f1=0.913 bacc=0.916
TEST: {'model': 'resnet18', 'mode': 'linear_probe', 'test_loss': 0.2296549982967831, 'test_acc': np.float64(0.9206349206349206), 'test_f1_macro': 0.9195304995488688, 'test_bacc': 0.9200597831725282}

=== resnet18 | finetune_partial

100.0%


[01/8] train loss=0.6275 acc=0.770 f1=0.769 | val loss=0.3658 acc=0.902 f1=0.902 bacc=0.907
[02/8] train loss=0.3388 acc=0.895 f1=0.894 | val loss=0.2679 acc=0.918 f1=0.919 bacc=0.921
[03/8] train loss=0.2638 acc=0.920 f1=0.920 | val loss=0.2431 acc=0.925 f1=0.926 bacc=0.928
[04/8] train loss=0.2373 acc=0.923 f1=0.923 | val loss=0.2310 acc=0.913 f1=0.915 bacc=0.917
[05/8] train loss=0.2324 acc=0.926 f1=0.926 | val loss=0.2121 acc=0.948 f1=0.949 bacc=0.949
[06/8] train loss=0.2150 acc=0.928 f1=0.928 | val loss=0.2085 acc=0.938 f1=0.940 bacc=0.940
[07/8] train loss=0.2223 acc=0.931 f1=0.931 | val loss=0.2073 acc=0.932 f1=0.933 bacc=0.935
[08/8] train loss=0.2151 acc=0.926 f1=0.926 | val loss=0.2011 acc=0.936 f1=0.938 bacc=0.938
TEST: {'model': 'densenet121', 'mode': 'linear_probe', 'test_loss': 0.23447569234976692, 'test_acc': np.float64(0.9387755102040817), 'test_f1_macro': 0.937449468789901, 'test_bacc': 0.9409817321990525}

=== densenet121 | finetune_partial ===
[01/12] train loss=0.4

100.0%


[01/8] train loss=0.3537 acc=0.884 f1=0.885 | val loss=0.1288 acc=0.957 f1=0.958 bacc=0.959
[02/8] train loss=0.1377 acc=0.955 f1=0.955 | val loss=0.1164 acc=0.957 f1=0.958 bacc=0.959
[03/8] train loss=0.1118 acc=0.962 f1=0.963 | val loss=0.1042 acc=0.966 f1=0.967 bacc=0.968
[04/8] train loss=0.0816 acc=0.971 f1=0.971 | val loss=0.1080 acc=0.964 f1=0.965 bacc=0.965
[05/8] train loss=0.0529 acc=0.984 f1=0.985 | val loss=0.0879 acc=0.964 f1=0.965 bacc=0.966
[06/8] train loss=0.0401 acc=0.987 f1=0.987 | val loss=0.0856 acc=0.966 f1=0.967 bacc=0.968
[07/8] train loss=0.0383 acc=0.988 f1=0.988 | val loss=0.0849 acc=0.968 f1=0.969 bacc=0.970
[08/8] train loss=0.0340 acc=0.990 f1=0.990 | val loss=0.1015 acc=0.966 f1=0.967 bacc=0.968
TEST: {'model': 'efficientnet_b0', 'mode': 'linear_probe', 'test_loss': 0.20166395903647352, 'test_acc': np.float64(0.9501133786848073), 'test_f1_macro': 0.9488003468132838, 'test_bacc': 0.951346958536501}

=== efficientnet_b0 | finetune_partial ===
[01/12] train 

100.0%


[01/8] train loss=0.4504 acc=0.847 f1=0.847 | val loss=0.2461 acc=0.925 f1=0.926 bacc=0.930
[02/8] train loss=0.2117 acc=0.929 f1=0.929 | val loss=0.1650 acc=0.948 f1=0.949 bacc=0.950
[03/8] train loss=0.1718 acc=0.941 f1=0.941 | val loss=0.1471 acc=0.957 f1=0.958 bacc=0.960
[04/8] train loss=0.1452 acc=0.955 f1=0.955 | val loss=0.1477 acc=0.952 f1=0.953 bacc=0.956
[05/8] train loss=0.1345 acc=0.955 f1=0.955 | val loss=0.1259 acc=0.957 f1=0.958 bacc=0.959
[06/8] train loss=0.1246 acc=0.962 f1=0.963 | val loss=0.1233 acc=0.964 f1=0.965 bacc=0.966
[07/8] train loss=0.1218 acc=0.961 f1=0.962 | val loss=0.1222 acc=0.964 f1=0.965 bacc=0.966
[08/8] train loss=0.1240 acc=0.965 f1=0.966 | val loss=0.1219 acc=0.964 f1=0.965 bacc=0.966
TEST: {'model': 'convnext_tiny', 'mode': 'linear_probe', 'test_loss': 0.14939166395977782, 'test_acc': np.float64(0.9501133786848073), 'test_f1_macro': 0.9491358260086948, 'test_bacc': 0.9492193173565724}

=== convnext_tiny | finetune_partial ===
[01/12] train los

100.0%


[01/8] train loss=0.4333 acc=0.841 f1=0.840 | val loss=0.2549 acc=0.923 f1=0.924 bacc=0.927
[02/8] train loss=0.2143 acc=0.926 f1=0.926 | val loss=0.1926 acc=0.950 f1=0.951 bacc=0.952
[03/8] train loss=0.1712 acc=0.944 f1=0.944 | val loss=0.1666 acc=0.943 f1=0.946 bacc=0.945
[04/8] train loss=0.1497 acc=0.952 f1=0.952 | val loss=0.1544 acc=0.950 f1=0.952 bacc=0.952
[05/8] train loss=0.1371 acc=0.956 f1=0.956 | val loss=0.1504 acc=0.950 f1=0.952 bacc=0.952
[06/8] train loss=0.1293 acc=0.962 f1=0.962 | val loss=0.1432 acc=0.941 f1=0.943 bacc=0.942
[07/8] train loss=0.1250 acc=0.961 f1=0.961 | val loss=0.1398 acc=0.945 f1=0.947 bacc=0.947
[08/8] train loss=0.1224 acc=0.965 f1=0.965 | val loss=0.1394 acc=0.948 f1=0.950 bacc=0.950
TEST: {'model': 'vit_b_16', 'mode': 'linear_probe', 'test_loss': 0.18393739098323986, 'test_acc': np.float64(0.9501133786848073), 'test_f1_macro': 0.9490709094989039, 'test_bacc': 0.9499113843068091}

=== vit_b_16 | finetune_partial ===
[01/12] train loss=0.9821 a